# LiteIE — LOLv2 Real trên Kaggle
Bật **GPU** và **Internet**, rồi Add Input dataset LOLv2. Chỉ sửa `DATASET_ROOT` bên dưới và Run All.
Đây là supervised L1 baseline bổ sung, không phải training recipe chính thức của paper.
Repo phải truy cập được từ Kaggle (public hoặc bạn tự cấu hình xác thực Git).


In [ ]:
DATASET_ROOT = "/kaggle/input/your-lolv2-dataset"  # Chỉ sửa dòng này


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

REPO_URL = "https://github.com/mluu59990-collab/LiteIE.git"
REPO_DIR = Path("/kaggle/working/LiteIE")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, f"Unexpected repository: {origin}"
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())
assert torch.cuda.is_available(), "Hãy bật GPU trong Settings của notebook"
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-train.txt")], check=True)


In [ ]:
# Kiểm tra cấu trúc, khả năng đọc ảnh và ghép cặp trước khi train.
command = [sys.executable, "-u", str(REPO_DIR / "train.py"),
           "--dataset-root", DATASET_ROOT, "--variant", "real"]
subprocess.run(command + ["--check-data"], cwd=REPO_DIR, check=True)


In [ ]:
# Mặc định: 200 epoch, batch 8, crop 256, Adam lr=1e-4, seed=42.
# Tự tiếp tục nếu checkpoint còn trong /kaggle/working.
# Sau khi đổi session: Add Input output cũ rồi đặt RESUME tới last.pt.
OUTPUT_DIR = Path("/kaggle/working/liteie-output")
RESUME = OUTPUT_DIR / "last.pt"
train_command = command + ["--output-dir", str(OUTPUT_DIR)]
if RESUME.exists():
    checkpoint = torch.load(RESUME, map_location="cpu", weights_only=True)
    if checkpoint["epoch"] < 200:
        train_command += ["--resume", str(RESUME)]
        subprocess.run(train_command, cwd=REPO_DIR, check=True)
    else:
        print("Đã hoàn tất 200 epoch. Xem kết quả bên dưới.")
else:
    subprocess.run(train_command, cwd=REPO_DIR, check=True)


In [ ]:
import json
import shutil
metrics = OUTPUT_DIR / "test_metrics.json"
if metrics.exists():
    print(json.dumps(json.loads(metrics.read_text()), indent=2))
print("Artifacts:", [p.name for p in OUTPUT_DIR.iterdir()])
shutil.make_archive("/kaggle/working/liteie-results", "zip", OUTPUT_DIR)
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/liteie-results.zip"))


## Kết quả
- `best.pth`: state dict dùng với `model.LiteIE`; có thể dùng trong inference gốc bằng cách đổi `MODEL_PATH`.
- `last.pt`: model, optimizer, best weights và epoch để resume.
- `config.json`, `splits.json`, `history.json`, `test_metrics.json` để tái kiểm tra thí nghiệm.
- Validation lấy 10% Train theo seed; Test chỉ dùng sau khi train. PSNR là trung bình theo ảnh RGB [0,1], không chỉnh độ sáng theo ground truth.
Lưu output notebook trước khi phiên Kaggle kết thúc để không mất checkpoint.
